# Automated Data Agent Evaluation



This notebook records, validates, scores, and exports the six-question challenge. Participant scoring is deterministic and remains separate from the optional facilitator-only SDK capture in the final section.



Each question is worth four points in each phase: original answer, selected source, paraphrase consistency, and reviewed query logic.

In [ ]:
# 1. Configure Evaluation Files and Run Mode

import json

import math

import os

import re

from datetime import datetime, timezone

from pathlib import Path



import pandas as pd



PARTICIPANT_ID = ""

RUN_MODE = "participant"  # participant scoring; SDK automation is separately opt-in below

EVALUATION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

OUTPUT_DIRECTORY = Path(".")

OUTPUT_PREFIX = "data_agent_evaluation"

CSV_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_scorecard.csv"

JSON_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_report.json"

KNOWN_FABRIC_ITEMS = {"LegalFirmSemanticModel", "LegalFirmDemo"}

MAX_TOTAL = 24.0


In [ ]:
# 2. Define the Six Question and Paraphrase Pairs

CHALLENGE = [

    {"id": "HC001", "question": "How many active clients do we have?", "paraphrase": "What is our current active customer count?", "expected_answer": 101, "expected_source": "LegalFirmSemanticModel", "expected_logic": "COUNT base_customers WHERE status = 'Active'"},

    {"id": "HC002", "question": "What is the total value of all matters?", "paraphrase": "What is our complete case portfolio worth?", "expected_answer": 123590881, "expected_source": "LegalFirmSemanticModel", "expected_logic": "SUM base_cases.case_value_gbp"},

    {"id": "HC003", "question": "How much revenue have we generated?", "paraphrase": "What is the total amount invoiced?", "expected_answer": 5420217, "expected_source": "LegalFirmSemanticModel", "expected_logic": "SUM base_transactions.amount_gbp WHERE transaction_type = 'Invoice'"},

    {"id": "HC004", "question": "How many invoices remain unpaid?", "paraphrase": "What is our unpaid invoice count?", "expected_answer": 54, "expected_source": "LegalFirmSemanticModel", "expected_logic": "COUNT base_transactions WHERE transaction_type = 'Invoice' AND payment_status = 'Unpaid'"},

    {"id": "HC005", "question": "How many legal cases are currently open?", "paraphrase": "How many open matters are on our books?", "expected_answer": 180, "expected_source": "LegalFirmSemanticModel", "expected_logic": "COUNT base_cases WHERE case_status = 'Open'"},

    {"id": "HC006", "question": "How many customers do we have?", "paraphrase": "What is our total client count?", "expected_answer": 171, "expected_source": "LegalFirmSemanticModel", "expected_logic": "COUNT base_customers"},

]

challenge_by_id = {item["id"]: item for item in CHALLENGE}

assert len(challenge_by_id) == 6


In [ ]:
# 3. Enter Baseline and Final Test Results

def blank_phase():

    return {

        "original_answer": "",

        "paraphrase_answer": "",

        "selected_source": "",

        "query_evidence": "",  # paste generated SQL/DAX or a concise copied run-step excerpt

        "logic_correct": None,  # set True only after table/measure, filters, and aggregation are verified

        "answer_observation": "",

        "logic_observation": "",

    }



OBSERVATIONS = [

    {"id": item["id"], "baseline": blank_phase(), "final": blank_phase()}

    for item in CHALLENGE

]



# Edit OBSERVATIONS above or update entries here before running the remaining cells.

OBSERVATIONS


In [ ]:
# 4. Validate Sources and Query Logic

def validate_observations(observations):

    issues = []

    ids = [entry.get("id") for entry in observations]

    if set(ids) != set(challenge_by_id) or len(ids) != 6:

        issues.append("Observations must contain each HC001-HC006 ID exactly once.")

    for entry in observations:

        test_id = entry.get("id", "unknown")

        for phase_name in ("baseline", "final"):

            phase = entry.get(phase_name, {})

            for field in ("original_answer", "paraphrase_answer", "selected_source"):

                if not str(phase.get(field, "")).strip():

                    issues.append(f"{test_id} {phase_name}: missing {field}.")

            source = str(phase.get("selected_source", "")).strip()

            if source and source not in KNOWN_FABRIC_ITEMS:

                issues.append(f"{test_id} {phase_name}: unknown Fabric item '{source}'.")

            if not str(phase.get("query_evidence", "")).strip():

                issues.append(f"{test_id} {phase_name}: add copied SQL/DAX evidence.")

            logic = phase.get("logic_correct")

            if logic not in (True, False, None):

                issues.append(f"{test_id} {phase_name}: logic_correct must be True, False, or None.")

            if logic is None:

                issues.append(f"{test_id} {phase_name}: complete the query-logic review.")

            if logic is True and not str(phase.get("query_evidence", "")).strip():

                issues.append(f"{test_id} {phase_name}: logic cannot be True without query evidence.")

    return issues



validation_issues = validate_observations(OBSERVATIONS)

validation_df = pd.DataFrame({"actionable_issue": validation_issues})

print(f"Validation found {len(validation_issues)} actionable issue(s).")

display(validation_df)


In [ ]:
# 5. Calculate Deterministic Scores

NUMBER_PATTERN = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")



def answer_matches(actual, expected):

    if isinstance(actual, (int, float)) and not isinstance(actual, bool):

        return math.isclose(float(actual), float(expected), rel_tol=0.0, abs_tol=0.01)

    matches = NUMBER_PATTERN.findall(str(actual).strip())

    return len(matches) == 1 and math.isclose(

        float(matches[0].replace(",", "")), float(expected), rel_tol=0.0, abs_tol=0.01

    )



def score_phase(phase, expected):

    checks = {

        "answer": answer_matches(phase.get("original_answer"), expected["expected_answer"]),

        "source": str(phase.get("selected_source", "")).strip().casefold() == expected["expected_source"].casefold(),

        "consistency": answer_matches(phase.get("paraphrase_answer"), expected["expected_answer"]),

        "logic": phase.get("logic_correct") is True and bool(str(phase.get("query_evidence", "")).strip()),

    }

    return checks, float(sum(checks.values()))



scored_rows = []

for entry in OBSERVATIONS:

    expected = challenge_by_id[entry["id"]]

    baseline_checks, baseline_score = score_phase(entry["baseline"], expected)

    final_checks, final_score = score_phase(entry["final"], expected)

    scored_rows.append({

        "id": entry["id"], "question": expected["question"], "paraphrase": expected["paraphrase"],

        "expected_answer": expected["expected_answer"], "expected_source": expected["expected_source"],

        "baseline_score": baseline_score, "final_score": final_score, "change": final_score - baseline_score,

        "baseline_source": entry["baseline"].get("selected_source"), "final_source": entry["final"].get("selected_source"),

        "baseline_logic": entry["baseline"].get("logic_correct"), "final_logic": entry["final"].get("logic_correct"),

        "baseline_checks": baseline_checks, "final_checks": final_checks,

        "baseline_query_evidence": entry["baseline"].get("query_evidence", ""),

        "final_query_evidence": entry["final"].get("query_evidence", ""),

    })



scorecard_df = pd.DataFrame(scored_rows)

baseline_total = float(scorecard_df["baseline_score"].sum())

final_total = float(scorecard_df["final_score"].sum())

assert 0.0 <= baseline_total <= MAX_TOTAL and 0.0 <= final_total <= MAX_TOTAL


In [ ]:
# 6. Build the Per-Question Scorecard

scorecard_df["validation_status"] = [

    "Ready" if row["baseline_score"] == 4 and row["final_score"] == 4 else "Review"

    for _, row in scorecard_df.iterrows()

]

scorecard_df["actionable_issue"] = [

    ", ".join(

        f"{phase} {name}"

        for phase in ("baseline", "final")

        for name, passed in row[f"{phase}_checks"].items()

        if not passed

    )

    for _, row in scorecard_df.iterrows()

]

display(scorecard_df[[

    "id", "question", "baseline_score", "final_score", "change",

    "baseline_source", "final_source", "baseline_logic", "final_logic",

    "validation_status", "actionable_issue",

]])


In [ ]:
# 7. Compare Baseline and Final Totals

improvement = final_total - baseline_total

percentage_improvement = None if baseline_total == 0 else 100.0 * improvement / baseline_total

regressions = scorecard_df.loc[scorecard_df["change"] < 0, "id"].tolist()

summary = {

    "baseline_score": baseline_total, "baseline_max": MAX_TOTAL,

    "final_score": final_total, "final_max": MAX_TOTAL,

    "absolute_improvement": improvement,

    "percentage_improvement": percentage_improvement,

    "question_regressions": regressions,

    "actionable_issue_count": len(validation_issues),

}

display(pd.DataFrame([summary]))


In [ ]:
# 8. Export CSV and JSON Evidence

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

export_df = scorecard_df.copy()

for column in ("baseline_checks", "final_checks"):

    export_df[column] = export_df[column].map(json.dumps)

export_df.to_csv(CSV_PATH, index=False)



report = {

    "metadata": {

        "participant_id": PARTICIPANT_ID,

        "evaluation_timestamp": EVALUATION_TIMESTAMP,

        "run_mode": RUN_MODE,

    },

    "summary": summary,

    "validation_issues": validation_issues,

    "observations": OBSERVATIONS,

    "scorecard": scored_rows,

}

JSON_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("CSV:", CSV_PATH.resolve())

print("JSON:", JSON_PATH.resolve())


In [ ]:
# 9. Validate Submission Artifacts

reloaded_csv = pd.read_csv(CSV_PATH)

reloaded_report = json.loads(JSON_PATH.read_text(encoding="utf-8"))

required_csv_columns = {"id", "baseline_score", "final_score", "baseline_query_evidence", "final_query_evidence"}

artifact_checks = {

    "CSV exists": CSV_PATH.is_file(),

    "JSON exists": JSON_PATH.is_file(),

    "CSV schema valid": required_csv_columns.issubset(reloaded_csv.columns),

    "Six question pairs present": set(reloaded_csv["id"]) == set(challenge_by_id),

    "Baseline total matches": math.isclose(float(reloaded_csv["baseline_score"].sum()), reloaded_report["summary"]["baseline_score"]),

    "Final total matches": math.isclose(float(reloaded_csv["final_score"].sum()), reloaded_report["summary"]["final_score"]),

    "Query evidence complete": all(

        str(phase.get("query_evidence", "")).strip()

        for item in OBSERVATIONS for phase in (item["baseline"], item["final"])

    ),

    "Screenshots or copied run-step evidence attached": False,  # set True before submission

}

submission_checklist_df = pd.DataFrame(

    [{"check": name, "complete": complete} for name, complete in artifact_checks.items()]

)

display(submission_checklist_df)


In [ ]:
# 10. Automate Independent SDK Evaluations (Facilitator Only)

RUN_SDK_AUTOMATION = False

SDK_AGENT_NAME = os.getenv("FABRIC_AGENT_NAME", "")

SDK_WORKSPACE_NAME = os.getenv("FABRIC_WORKSPACE_NAME", "") or None

SDK_TABLE_NAME = os.getenv("FABRIC_EVALUATION_TABLE", "automated_evaluation_output")

SDK_STAGE = os.getenv("FABRIC_DATA_AGENT_STAGE", "production")

SDK_RAW_CSV_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_sdk_raw.csv"

SDK_RAW_JSON_PATH = OUTPUT_DIRECTORY / f"{OUTPUT_PREFIX}_sdk_raw.json"



def redact_sensitive(value):

    text = str(value)

    return re.sub(

        r"(?i)(bearer|token|api[_ -]?key|secret)\s*[:=]\s*\S+",

        r"\1=[REDACTED]",

        text,

    )



if RUN_SDK_AUTOMATION:

    if not SDK_AGENT_NAME:

        raise ValueError("Set FABRIC_AGENT_NAME in the notebook environment before SDK automation.")

    import importlib

    import time



    fabric_evaluation = importlib.import_module("fabric.dataagent.evaluation")

    sdk_rows = [

        {"question": prompt, "expected_answer": item["expected_answer"]}

        for item in CHALLENGE

        for prompt in (item["question"], item["paraphrase"])

    ]

    sdk_input_df = pd.DataFrame(sdk_rows)

    for attempt in range(1, 4):

        try:

            sdk_evaluation_id = fabric_evaluation.evaluate_data_agent(

                sdk_input_df,

                SDK_AGENT_NAME,

                workspace_name=SDK_WORKSPACE_NAME,

                table_name=SDK_TABLE_NAME,

                data_agent_stage=SDK_STAGE,

            )

            break

        except Exception:

            if attempt == 3:

                raise

            time.sleep(2 ** attempt)



    sdk_details_df = fabric_evaluation.get_evaluation_details(

        evaluation_id=sdk_evaluation_id,

        table_name=SDK_TABLE_NAME,

        get_all_rows=True,

        verbose=False,

    )

    sdk_summary_df = fabric_evaluation.get_evaluation_summary(

        table_name=SDK_TABLE_NAME,

        verbose=False,

    )

    sensitive_columns = [

        column

        for column in sdk_details_df.columns

        if any(term in column.casefold() for term in ("token", "secret", "credential", "key"))

    ]

    redacted_details_df = sdk_details_df.drop(columns=sensitive_columns).map(redact_sensitive)

    redacted_details_df.to_csv(SDK_RAW_CSV_PATH, index=False)

    sdk_payload = {

        "evaluation_id": str(sdk_evaluation_id),

        "agent_name": SDK_AGENT_NAME,

        "workspace_name": SDK_WORKSPACE_NAME,

        "stage": SDK_STAGE,

        "summary": json.loads(sdk_summary_df.to_json(orient="records")),

        "details": json.loads(redacted_details_df.to_json(orient="records")),

    }

    SDK_RAW_JSON_PATH.write_text(json.dumps(sdk_payload, indent=2), encoding="utf-8")

    print("Independent SDK evidence saved separately:", SDK_RAW_CSV_PATH, SDK_RAW_JSON_PATH)

else:

    print("SDK automation is off. Participant-entered deterministic scoring remains authoritative.")
